# C11 — Têtes de classification de la famille B (Brique 4, R4, P3, P4, P8, E4, E7)

Ce notebook entraîne les têtes de la famille B sur les moyennes par couche calculées par C10
(datasets Kaggle privés `echolens-emb-wav2vec2-base`, `echolens-emb-mhubert-147`,
`echolens-emb-whisper-base`). Quatre modèles : wav2vec2-base, mhubert-147, whisper-base
(trames utiles) et whisper-base-padded (moyenne sur les 1 500 trames, pour mesurer l'effet du
remplissage). Deux tâches : ESC-50 (5 tours, P3) et les émotions (CREMA-D et RAVDESS réunis,
découpage par acteur de C4, P1).

La tête, en trois étapes : standardisation de chaque (couche, dimension) avec le train seul
(R4), somme pondérée des couches w = softmax(a), puis MLP. Tous les réglages sont dans
`configs/heads.yaml`. Tout le calcul vient du dépôt (`echolens.training.heads` et
`echolens.training.head_training`) ; le notebook enchaîne les étapes et vérifie les données.

Calcul sur **CPU** : le quota GPU est gardé pour C9. Les sorties (têtes, statistiques,
probabilités) restent dans les sorties du notebook, jamais sur GitHub ; seuls les deux JSON de
résultats vont dans `training/results/`.

## a. Démarrage

On clone le dépôt (branche `BRANCH`) et on l'installe en mode éditable (I9), **sans** l'extra
`[ml]` : torch est déjà installé sur Kaggle. L'installation éditable n'est visible qu'après un
redémarrage du noyau : on ajoute donc `src/` au `sys.path`.

In [ ]:
import sys

BRANCH = "main"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. torch sur CPU

Aucun GPU n'est demandé : les têtes sont petites (environ 200 000 paramètres) et les
représentations déjà calculées.

In [ ]:
import os

import torch

print("torch", torch.__version__)
print("GPU visible :", torch.cuda.is_available(), "(non utilisé)")
print("Cœurs CPU :", os.cpu_count(), "· threads torch :", torch.get_num_threads())

## c. Trouver les représentations de C10

On cherche chaque `metadata.json` sous `/kaggle/input` : il dit l'encodeur, le jeu et s'il
s'agit de la version avec remplissage. On attend 12 dossiers : 3 encodeurs × 3 jeux, plus les
3 jeux de Whisper avec remplissage.

In [ ]:
from pathlib import Path

from echolens.training.embeddings import METADATA_FILE, EmbeddingMetadata

INPUT_DIR = Path("/kaggle/input")
EXPECTED_FOLDERS = 12

embedding_dirs = {}
for metadata_path in sorted(INPUT_DIR.glob(f"**/{METADATA_FILE}")):
    metadata_text = metadata_path.read_text(encoding="utf-8")
    metadata = EmbeddingMetadata.model_validate_json(metadata_text)
    key = (metadata.encoder_id, metadata.padded, metadata.dataset)
    embedding_dirs[key] = metadata_path.parent
for key, folder in sorted(embedding_dirs.items()):
    print(key, "→", folder)
if len(embedding_dirs) != EXPECTED_FOLDERS:
    raise RuntimeError(f"{len(embedding_dirs)} dossiers trouvés au lieu de {EXPECTED_FOLDERS}")

## d. Manifestes, découpages et contrôles

On lit les manifestes versionnés, le découpage des acteurs de C4
(`data/splits/emotion_split.csv`) et les 6 classes communes (P2). Chaque fichier de
représentations est relu avec un contrôle : son `index.csv` doit être exactement le manifeste,
dans le même ordre.

Contrôles avant tout entraînement :
- aucun acteur commun entre train, validation et test (P1) ;
- aucune classe hors des 6 émotions (calme et surprise exclus, P2).

In [ ]:
import numpy as np

from echolens.config import CONFIG_DIR, REPO_ROOT
from echolens.core.data.emotions import load_emotion_labels
from echolens.core.data.manifest import load_manifest
from echolens.core.data.splits import check_no_speaker_leak, load_speaker_splits, load_split_config
from echolens.training.embeddings import load_embeddings, manifest_index
from echolens.training.esc50_svm import SvmConfig
from echolens.training.evaluation import EvaluationConfig
from echolens.training.heads import (
    check_classes,
    check_disjoint_speakers,
    emotion_examples,
    emotion_part,
    esc50_targets_and_folds,
    load_heads_config,
)

heads_config = load_heads_config()
evaluation_config = EvaluationConfig.from_yaml(CONFIG_DIR / "evaluation.yaml")
category_names = SvmConfig.from_yaml(CONFIG_DIR / "baseline_svm.yaml").category_names
labels = load_emotion_labels()
split_config = load_split_config()
speaker_splits = load_speaker_splits(REPO_ROOT / "data" / "splits" / "emotion_split.csv")
check_no_speaker_leak(speaker_splits)

manifests = {}
for dataset_id in ["esc50", *heads_config.emotion_datasets]:
    manifests[dataset_id] = load_manifest(REPO_ROOT / "data" / "manifests" / f"{dataset_id}.csv")
    print(dataset_id, ":", len(manifests[dataset_id]), "lignes")


def load_features(model, dataset_id):
    # Représentations d'un modèle sur un jeu, après contrôle de l'ordre des lignes.
    folder = embedding_dirs[(model.encoder, model.padded, dataset_id)]
    files = load_embeddings(folder)
    expected_index = manifest_index(manifests[dataset_id])
    if files.index != expected_index:
        raise RuntimeError(f"{folder} : index différent du manifeste")
    if not np.all(np.isfinite(files.embeddings)):
        raise RuntimeError(f"{folder} : valeurs NaN ou infinies")
    return files.embeddings


examples_by_dataset = {}
for dataset_id in heads_config.emotion_datasets:
    rows = manifests[dataset_id]
    examples_by_dataset[dataset_id] = emotion_examples(rows, labels, speaker_splits, split_config)

# Contrôles sur des représentations factices : les parties ne dépendent pas du modèle.
placeholder = {}
for dataset_id in heads_config.emotion_datasets:
    placeholder[dataset_id] = np.zeros((len(manifests[dataset_id]), 1, 1), dtype=np.float32)
SPLIT_NAMES = ["train", "validation", "test"]
check_parts = {}
for split_name in SPLIT_NAMES:
    check_parts[split_name] = emotion_part(examples_by_dataset, placeholder, split_name)
check_disjoint_speakers(check_parts)
for split_name, part in check_parts.items():
    check_classes(part.data.targets, len(labels.emotions))
    speaker_count = len(set(part.speakers))
    class_counts = np.bincount(part.data.targets, minlength=len(labels.emotions)).tolist()
    print(f"{split_name} : {len(part.speakers)} extraits, {speaker_count} acteurs, {class_counts}")
print("Aucun acteur commun entre les parties ; toutes les classes sont parmi", labels.emotions)

esc50_targets, esc50_folds = esc50_targets_and_folds(manifests["esc50"])
check_classes(esc50_targets, heads_config.esc50_class_count)
print("ESC-50 : extraits par fold", np.bincount(esc50_folds)[1:].tolist())

## e. Émotions

Pour chaque modèle : statistiques de standardisation sur le train seul (R4) ; réglage de la
grille avec la graine 42 sur la validation ; puis une tête par graine (42, 43, 44), entraînée
sur la courbe complète (100 époques, sans arrêt anticipé), dont on garde l'époque au meilleur F1
de validation. Le test n'est prédit qu'une fois par tête.

Sorties par modèle : `/kaggle/working/c11/emotions/<modèle>/` avec la standardisation, les
têtes (`head_seed<graine>.pt`) et les probabilités de validation et de test (pour le
calibrage de C13).

In [ ]:
import time
from dataclasses import asdict

from echolens.training.head_training import run_emotions, save_head
from echolens.training.heads import (
    emotion_results,
    predicted_classes,
    run_of_seed,
    save_probabilities,
    save_standardization,
)

OUTPUT_DIR = Path("/kaggle/working/c11")
emotion_results_by_model = {}
emotion_statistics_predictions = {}
emotion_seconds = {}

for model in heads_config.models:
    start = time.perf_counter()
    features_by_dataset = {}
    for dataset_id in heads_config.emotion_datasets:
        features_by_dataset[dataset_id] = load_features(model, dataset_id)
    parts = {}
    for split_name in SPLIT_NAMES:
        parts[split_name] = emotion_part(examples_by_dataset, features_by_dataset, split_name)
    class_count = len(labels.emotions)
    run = run_emotions(
        parts["train"].data, parts["validation"].data, parts["test"].data, heads_config, class_count
    )
    # Sauvegarde : standardisation, têtes, probabilités de validation et de test.
    model_dir = OUTPUT_DIR / "emotions" / model.id
    save_standardization(model_dir / "standardization.npz", run.heads[0].stats)
    for head, validation_run in zip(run.heads, run.validation_predictions, strict=True):
        seed = head.predictions.seed
        save_head(model_dir / f"head_seed{seed}.pt", head.model)
        validation_targets = parts["validation"].data.targets
        validation_path = model_dir / f"validation_seed{seed}.npz"
        save_probabilities(validation_path, validation_targets, validation_run.probabilities)
        test_path = model_dir / f"test_seed{seed}.npz"
        save_probabilities(test_path, parts["test"].data.targets, head.predictions.probabilities)
    seed_runs = [head.predictions for head in run.heads]
    test_part = parts["test"]
    results = emotion_results(test_part, seed_runs, class_count, heads_config, evaluation_config)
    results["tuning"] = [asdict(outcome) for outcome in run.outcomes]
    results["chosen"] = asdict(run.chosen)
    statistics_run = run_of_seed(seed_runs, heads_config.statistics_seed)
    emotion_statistics_predictions[model.id] = predicted_classes(statistics_run.probabilities)
    seconds = time.perf_counter() - start
    emotion_seconds[model.id] = seconds
    results["compute_seconds"] = seconds
    emotion_results_by_model[model.id] = results
    f1 = results["macro_f1"]
    print(f"{model.id} : F1 macro {f1['mean']:.3f} ± {f1['std']:.3f}, {seconds:.0f} s")
emotion_test_part = parts["test"]

## f. ESC-50

Pour chaque modèle et chaque tour k (P3) : réglage sur 3 folds (graine 42), choix du réglage
et du nombre d'époques sur le fold k + 1 ; puis, avec ces réglages fixés, une tête par graine
entraînée sur les 4 folds hors test pendant ce nombre d'époques, sans validation, et
évaluée une seule fois sur le fold k. Le réglage parcourt la courbe complète (100 époques,
sans arrêt anticipé) avant de choisir l'époque.

Sorties par modèle : `/kaggle/working/c11/esc50/<modèle>/fold<k>/`.

In [ ]:
from echolens.training.head_training import run_esc50_round
from echolens.training.heads import Esc50FoldPredictions, esc50_results, pooled_esc50_predictions

esc50_results_by_model = {}
esc50_statistics_predictions = {}
esc50_seconds = {}
fold_numbers = sorted(set(esc50_folds.tolist()))

for model in heads_config.models:
    start = time.perf_counter()
    features = load_features(model, "esc50")
    fold_predictions = []
    tuning_by_fold = []
    for test_fold in fold_numbers:
        round_run = run_esc50_round(features, esc50_targets, esc50_folds, test_fold, heads_config)
        fold_dir = OUTPUT_DIR / "esc50" / model.id / f"fold{test_fold}"
        save_standardization(fold_dir / "standardization.npz", round_run.heads[0].stats)
        validation_probabilities = round_run.validation_predictions.probabilities
        validation_targets = round_run.validation.targets
        validation_path = fold_dir / "validation.npz"
        save_probabilities(validation_path, validation_targets, validation_probabilities)
        for head in round_run.heads:
            seed = head.predictions.seed
            save_head(fold_dir / f"head_seed{seed}.pt", head.model)
            test_path = fold_dir / f"test_seed{seed}.npz"
            save_probabilities(test_path, round_run.test_targets, head.predictions.probabilities)
        fold_predictions.append(
            Esc50FoldPredictions(
                test_fold=test_fold,
                setting=round_run.chosen.setting,
                epochs=round_run.chosen.best_epoch,
                y_true=round_run.test_targets,
                runs=[head.predictions for head in round_run.heads],
            )
        )
        tuning = {"test_fold": test_fold, "outcomes": [asdict(item) for item in round_run.outcomes]}
        tuning_by_fold.append(tuning)
    results = esc50_results(fold_predictions, heads_config, len(category_names), evaluation_config)
    results["tuning"] = tuning_by_fold
    esc50_y_true, statistics_pred = pooled_esc50_predictions(
        fold_predictions, heads_config.statistics_seed
    )
    esc50_statistics_predictions[model.id] = statistics_pred
    seconds = time.perf_counter() - start
    esc50_seconds[model.id] = seconds
    results["compute_seconds"] = seconds
    esc50_results_by_model[model.id] = results
    f1 = results["macro_f1"]
    print(f"{model.id} : F1 macro {f1['mean']:.3f} ± {f1['std']:.3f}, {seconds:.0f} s")

## g. McNemar et JSON de résultats

McNemar (P8) sur les prédictions de la graine 42 uniquement (décidé à l'avance), sur les mêmes
extraits de test : pour ESC-50, les 5 folds de test réunis (2 000 extraits, chacun prédit une
fois) ; pour les émotions, le test de CREMA-D et de RAVDESS réunis.

Deux JSON (→ `training/results/`) : `c11_family_b_esc50.json` et `c11_family_b_emotions.json`.

In [ ]:
import datetime
import json
import subprocess

from echolens.training.heads import mcnemar_results, write_results

RESULTS_DIR = Path("/kaggle/working/results")
commit = subprocess.run(
    ["git", "-C", CLONE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
svm_reference_path = REPO_ROOT / "training" / "results" / "esc50_mfcc_svm.json"
svm_reference = json.loads(svm_reference_path.read_text(encoding="utf-8"))


def header(experiment):
    return {
        "experiment": experiment,
        "date": datetime.datetime.now(datetime.UTC).isoformat(),
        "branch": BRANCH,
        "commit": commit,
        "torch_version": torch.__version__,
        "device": "cpu",
        "heads_config": heads_config.model_dump(),
    }


emotion_mcnemar = mcnemar_results(
    emotion_test_part.data.targets,
    emotion_statistics_predictions,
    heads_config.mcnemar_pairs,
    evaluation_config,
)
emotion_content = header("c11_family_b_emotions")
emotion_content["classes"] = labels.emotions
emotion_content["test_size"] = len(emotion_test_part.data.targets)
emotion_content["models"] = emotion_results_by_model
emotion_content["mcnemar"] = emotion_mcnemar
write_results(RESULTS_DIR / "c11_family_b_emotions.json", emotion_content)

esc50_mcnemar = mcnemar_results(
    esc50_y_true, esc50_statistics_predictions, heads_config.mcnemar_pairs, evaluation_config
)
esc50_content = header("c11_family_b_esc50")
esc50_content["category_names"] = category_names
esc50_content["svm_reference"] = {
    "macro_f1_mean": svm_reference["macro_f1_mean"],
    "macro_f1_std": svm_reference["macro_f1_std"],
}
esc50_content["models"] = esc50_results_by_model
esc50_content["mcnemar"] = esc50_mcnemar
write_results(RESULTS_DIR / "c11_family_b_esc50.json", esc50_content)
print(sorted(path.name for path in RESULTS_DIR.iterdir()))

## h. Résumé

F1 macro ± écart-type (graines pour les émotions, 5 tours pour ESC-50), intervalle à 95 %
(graine 42), par jeu et par genre, puis McNemar et poids des couches.

In [ ]:
import pandas as pd


def summary_row(task, model_id, results):
    f1 = results["macro_f1"]
    low, high = results["macro_f1_ci"]
    return {
        "tâche": task,
        "modèle": model_id,
        "F1 macro": f"{f1['mean']:.3f} ± {f1['std']:.3f}",
        "IC 95 %": f"[{low:.3f}, {high:.3f}]",
        "secondes": round(results["compute_seconds"]),
    }


summary_rows = []
for model_id, results in esc50_results_by_model.items():
    summary_rows.append(summary_row("ESC-50", model_id, results))
for model_id, results in emotion_results_by_model.items():
    summary_rows.append(summary_row("émotions", model_id, results))
display(pd.DataFrame(summary_rows))

group_rows = []
for model_id, results in emotion_results_by_model.items():
    groups = {**results["macro_f1_by_dataset"], **results["macro_f1_by_gender"]}
    row = {"modèle": model_id}
    for name, summary in groups.items():
        row[name] = f"{summary['mean']:.3f} ± {summary['std']:.3f}"
    group_rows.append(row)
display(pd.DataFrame(group_rows))

for task, entries in [("ESC-50", esc50_mcnemar), ("émotions", emotion_mcnemar)]:
    print(task)
    display(pd.DataFrame(entries))

weight_rows = []
for task, by_model in [("ESC-50", esc50_results_by_model), ("émotions", emotion_results_by_model)]:
    for model_id, results in by_model.items():
        weights = [round(value, 3) for value in results["layer_weights"]["mean"]]
        weight_rows.append({"tâche": task, "modèle": model_id, "poids des couches": weights})
display(pd.DataFrame(weight_rows))

## Nettoyage

On supprime le clone du dépôt : les sorties ne gardent que `c11/` (têtes, statistiques,
probabilités) et `results/` (les deux JSON).

In [ ]:
import shutil

shutil.rmtree(CLONE_DIR, ignore_errors=True)
print(sorted(path.name for path in Path("/kaggle/working").iterdir()))